# Matched ROI sampling and KRONOS2 extraction


In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "config" / "project.yaml").is_file()
             and (path / "src" / "pathology").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within this repository.")
sys.path.insert(0, str(ROOT / "src"))
from pathology.config import load_paths
PATHS = load_paths(ROOT)
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
os.environ.setdefault("HF_MODULES_CACHE", str(ROOT / ".cache" / "huggingface" / "modules"))
os.environ.setdefault("TORCH_HOME", str(ROOT / ".cache" / "torch"))


## Configuration


In [ ]:
import json
import subprocess
import pandas as pd
import yaml
from pathology.config import kronos_config
PIPELINE = ROOT / 'src' / 'pathology' / 'kronos_pipeline.py'
RUN = PATHS.run
RUN.mkdir(parents=True, exist_ok=True)
CONFIG = RUN / 'extraction_config.yaml'
CONFIG.write_text(yaml.safe_dump(kronos_config(ROOT, PATHS), sort_keys=False), encoding='utf-8')
CASE_IDS = []
MAX_PATCHES_PER_CASE = None


## Preflight


In [ ]:
command = [sys.executable, str(PIPELINE), '--config', str(CONFIG), '--stage', 'preflight']
result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
if not result.stdout.strip():
    raise RuntimeError(result.stderr or 'Preflight returned no report')
preflight = json.loads(result.stdout)
if preflight['blockers']:
    raise RuntimeError(preflight['blockers'])
result.check_returncode()
pd.Series({key: preflight[key] for key in ['selected_cases', 'enabled_marker_count', 'weights_bytes']})


## Resumable extraction


In [ ]:
command = [sys.executable, str(PIPELINE), '--config', str(CONFIG), '--stage', 'extract', '--resume']
for case_id in CASE_IDS:
    command.extend(['--case-id', case_id])
if MAX_PATCHES_PER_CASE is not None:
    command.extend(['--max-patches-per-case', str(MAX_PATCHES_PER_CASE)])
with (RUN / 'extraction.log').open('w', encoding='utf-8') as log:
    subprocess.run(command, cwd=ROOT, stdout=log, stderr=subprocess.STDOUT, check=True)


## Extraction summary


In [ ]:
manifest = json.loads((RUN / 'run_manifest.json').read_text(encoding='utf-8'))
wsi_summary = pd.read_csv(RUN / 'wsi_summary.csv')
wsi_summary[['case_id', 'registration_status', 'roi_count', 'candidate_patch_count', 'patch_count', 'skipped_patch_count', 'feature_dim', 'elapsed_seconds']]
